In [1]:
import os
from dotenv import load_dotenv

from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings

from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec
from langchain_pinecone import PineconeVectorStore

from langchain_openai import ChatOpenAI
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\langchain_pinecone\__init__.py:3: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain_core.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain_core.pydantic_v1 import BaseModel`
with: `from pydantic import BaseModel`
or the v1 compatibility namespace if you are working in a code base that has not been fully upgraded to pydantic 2 yet. 	from pydantic.v1 import BaseModel

  from langchain_pinecone.vectorstores import Pinecone, PineconeVectorStore


In [3]:
load_dotenv()
PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY")
DEEPSEEK_API_KEY = "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"

In [ ]:
#Load PDF and extract data
def load_pdf_file(data):
    loader=DirectoryLoader(data,
                           glob="*.pdf",
                           loader_cls=PyPDFLoader)
    
    documents=loader.load()

    return documents

extracted_data = load_pdf_file(data="../Data/")
print("Total Document: ",len(extracted_data))

Total Document:  637


In [6]:
#split document
def text_split(docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 500,
        chunk_overlap=20
    )
    text_chunks = text_splitter.split_documents(docs)
    return text_chunks

text_chunks = text_split(extracted_data)
print("Length of Text Chunks: ", len(text_chunks))

Length of Text Chunks:  5859


In [7]:
#Download embeddings model
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

C:\Users\user\AppData\Local\Temp\ipykernel_9236\3659953109.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(


In [8]:
#set up victor database called pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)
index_name = "medicalbot"

# Create index if not exists
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print(f"Index '{index_name}' created.")
else:
    print(f"Index '{index_name}' already exists.")

Index 'medicalbot' already exists.


In [9]:
#store embedding new data in pinecone(embed new data)
docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embedding,
    batch_size=32
)

#Or load in existing index(I already embeded data in pinecone)
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

retriever = docsearch.as_retriever()

In [10]:
docsearch

In [11]:
#searches stored data from pinecone and fetches the 3 most relevant passages
retriever = docsearch.as_retriever(search_type="similarity",search_kwargs={"k":3})
retrived_docs = retriever.invoke("What is Acne")
retrived_docs

[Document(id='5e6726ba-9232-4d04-8b04-baf882ba1e81', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 39.0, 'page_label': '40', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': '..\\Data\\Medical_book.pdf', 'total_pages': 637.0}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='4bc14399-3bf0-4857-842e-3fcbb8d70e44', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 39.0, 'page_label': '40', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Data\\Medical_book.pdf', 'total_pages': 637.0}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='8027f7e2-7f83-4aa3-b8a8-e67edb1f33d6', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 39

In [ ]:
#set up llm model
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="deepseek/deepseek-r1:free",
    api_key=DEEPSEEK_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    temperature=0.3
    #can add top k and top p to 
)

In [14]:
system_prompt=(
    "You are an assistant for medical question-answering tasks."
    "Use the following pieces of retrived context to answer the question"
    "If you don't know the answer, say that yhou don't know."
    "Use three sentences maximum and keep the answer concise.\n\n{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system",system_prompt),
        ("human","input"),
    ]
)

question_answer_chain = create_stuff_documents_chain(llm,prompt)
rag_chain = create_retrieval_chain(retriever,question_answer_chain)

In [15]:
question = "What is Diabete?"
response = rag_chain.invoke({"input":question})

print("Q :",question)
print("A :", response["answer"])

Q : What is Diabete?
A : Diabetes mellitus results from insufficient or ineffective insulin, leading to chronic hyperglycemia (high blood sugar), which can damage organs if untreated. Hypoglycemia (low blood sugar) is identified through blood sugar testing but is not directly caused by the disease itself, unlike hyperglycemia. Early diagnosis is critical to prevent complications.
